# Analise de Rede Social com Grafos - Dataset Facebook (SNAP)

**Disciplina:** Teoria e Aplicacao de Grafos, 2026/2
**Professor:** Dr Dibio Leandro
**Integrantes:** [Carlos Roberto - 251004385], [Gabrielly Batista - 251004400], [Helena Miranda - 251004302]

**Dataset:** [ego-Facebook (SNAP)](https://snap.stanford.edu/data/egonets-Facebook.html) -
J. McAuley e J. Leskovec, *Learning to Discover Social Circles in Ego Networks*, NIPS 2012.

## Objetivo

Identificar nos de maior influencia e comunidades em uma amostra de 404 usuarios do
grafo de amizades do Facebook, usando metricas de centralidade e deteccao de
comunidades

## Organizacao

O notebook esta dividido em 4 blocos. Cada etapa tem uma celula de especificacao
tecnica e uma celula de codigo vazia.

| Bloco | Etapas | Depende de |
|---|---|---|
| 1 - Dados e grafo | 1.1 Coleta, 1.2 Amostragem | - |
| 2 - Metricas | 2.1 Comunidades, 2.2 Centralidades | Bloco 1 |
| 3 - Visualizacoes | 3.1 Graficos | Bloco 2 |
| 4 - Analise | 4.1 Discussao | Blocos 2 e 3 |

Os blocos 2 e 3 podem ser divididos por metrica entre integrantes, desde que o
Bloco 1 ja esteja pronto.


## Bloco 1 - Dados e grafo

### 1.1 Coleta e carregamento

**Entrada:** `../data/facebook_combined.txt` (4039 nos, 88234 arestas, uma aresta
por linha, formato `no1 no2`).

**Implementar:**
- Carregar o arquivo com `nx.read_edgelist(caminho, nodetype=int)`;
- Reportar: numero de nos, numero de arestas, grau medio, conectividade
  (`nx.is_connected`).


In [1]:
import networkx as nx

CAMINHO_DADOS = "../data/facebook_combined.txt"

grafo_completo = nx.read_edgelist(CAMINHO_DADOS, nodetype=int)

print(f"Nos: {grafo_completo.number_of_nodes()}")
print(f"Arestas: {grafo_completo.number_of_edges()}")
print(f"Grau medio: {2 * grafo_completo.number_of_edges() / grafo_completo.number_of_nodes():.2f}")
print(f"Grafo completo conectado: {nx.is_connected(grafo_completo)}")


Nos: 4039
Arestas: 88234
Grau medio: 43.69
Grafo completo conectado: True


### 1.2 Amostragem dos 404 nos

**Requisito do enunciado:** 404 nos sorteados aleatoriamente, com todas as arestas
entre eles. A semente de sorteio deve ser propria do grupo e fixa, para
reprodutibilidade.

**Implementar:**
- `random.sample` sobre os nos do grafo completo, com semente fixa;
- Subgrafo induzido: `grafo_completo.subgraph(nos_amostrados)`;
- Reportar: numero de componentes conexos, tamanho do maior componente, numero de
  nos isolados;
- Salvar a amostra (arestas e lista de nos) em `../data/`, para reprodutibilidade
  entre execucoes.

**Observacao tecnica:** a amostragem aleatoria de nos tende a fragmentar o grafo,
porque a rede original e formada por clusters locais com poucas pontes entre si.
Esperado: multiplos componentes conexos e uma proporcao relevante de nos isolados.


In [ ]:
import random

SEMENTE = 73         #o grupo sempre chega no mesmo resultado
N_NOS_AMOSTRA = 404  # quantidade exigida pelo enunciado
CAMINHO_SUBGRAFO_ARESTAS = "../data/subgrafo_404_nos.edgelist"
CAMINHO_SUBGRAFO_NOS = "../data/subgrafo_404_nos.nodelist"

random.seed(SEMENTE)
nos_amostrados = random.sample(list(grafo_completo.nodes()), N_NOS_AMOSTRA)

#so entram arestas cujas DUAS pontas foram sorteadas
G = grafo_completo.subgraph(nos_amostrados).copy()

componentes = sorted(nx.connected_components(G), key=len, reverse=True)
isolados = [n for n in G.nodes() if G.degree(n) == 0]

print(f"Subgrafo amostrado: {G.number_of_nodes()} nos, {G.number_of_edges()} arestas")
print(f"Componentes conexos: {len(componentes)} (maior tem {len(componentes[0])} nos)")
print(f"Nos isolados (sem nenhum vizinho tambem amostrado): {len(isolados)}")

# write_edgelist nao salva nos isolados, guardamos
# a lista completa dos 404 nos num arquivo separado
nx.write_edgelist(G, CAMINHO_SUBGRAFO_ARESTAS, data=False)
with open(CAMINHO_SUBGRAFO_NOS, "w") as f:
    f.write("\n".join(str(n) for n in sorted(G.nodes())))


Subgrafo amostrado: 404 nos, 760 arestas
Componentes conexos: 113 (maior tem 95 nos)
Nos isolados (sem nenhum vizinho tambem amostrado): 87


## Bloco 2 - Metricas

### 2.1 Comunidades (Louvain)

**Implementar:**
- `nx.community.louvain_communities(G, seed=...)`;
- Reportar: numero de comunidades, tamanho das maiores.


In [3]:
particao = nx.community.louvain_communities(G, seed=SEMENTE)
tamanhos = sorted((len(c) for c in particao), reverse=True)

print(f"Comunidades encontradas: {len(particao)}")
print(f"Tamanhos das 5 maiores: {tamanhos[:5]}")


Comunidades encontradas: 118
Tamanhos das 5 maiores: [37, 36, 28, 28, 26]


### 2.2 Centralidades

**Implementar as quatro metricas pedidas no enunciado:**
- Grau: `nx.degree_centrality`;
- Intermediacao: `nx.betweenness_centrality`;
- Proximidade: `nx.closeness_centrality`;
- Autovetor: `nx.eigenvector_centrality` ou `nx.eigenvector_centrality_numpy`.

**Restricao tecnica:** proximidade e autovetor exigem grafo conexo. Se o subgrafo
amostrado estiver fragmentado (Bloco 1.2), calcular essas duas metricas sobre o
maior componente conexo e justificar a escolha.

**Reportar:** top 10 nos por metrica.


In [ ]:
# proximidade e autovetor exigem grafo conexo
# por isso usa so o maior componente, nao o subgrafo inteiro
maior_componente = G.subgraph(componentes[0]).copy()

grau = nx.degree_centrality(G)
intermediacao = nx.betweenness_centrality(G)
proximidade = nx.closeness_centrality(maior_componente)
autovetor = nx.eigenvector_centrality_numpy(maior_componente)

def top10(metrica):
    return sorted(metrica.items(), key=lambda item: item[1], reverse=True)[:10]

for nome, metrica in [
    ("Grau", grau),
    ("Intermediacao", intermediacao),
    ("Proximidade", proximidade),
    ("Autovetor", autovetor),
]:
    print(f"\nTop 10 - {nome}")
    for no, valor in top10(metrica):
        print(f"  no {no}: {valor:.4f}")



Top 10 - Grau
  no 0: 0.0893
  no 2543: 0.0794
  no 2206: 0.0496
  no 2409: 0.0496
  no 1979: 0.0471
  no 2201: 0.0422
  no 2526: 0.0422
  no 1917: 0.0422
  no 2069: 0.0397
  no 2090: 0.0372

Top 10 - Intermediacao
  no 563: 0.0163
  no 1574: 0.0160
  no 641: 0.0157
  no 1231: 0.0147
  no 1620: 0.0127
  no 484: 0.0119
  no 1688: 0.0089
  no 0: 0.0072
  no 2543: 0.0069
  no 1745: 0.0061

Top 10 - Proximidade
  no 563: 0.3287
  no 1574: 0.3219
  no 1688: 0.3123
  no 484: 0.3102
  no 1620: 0.3092
  no 1745: 0.3042
  no 641: 0.2831
  no 538: 0.2823
  no 1344: 0.2806
  no 962: 0.2789

Top 10 - Autovetor
  no 1211: 0.3741
  no 1399: 0.3603
  no 1491: 0.2886
  no 1191: 0.2840
  no 1662: 0.2808
  no 1554: 0.2662
  no 1823: 0.2657
  no 966: 0.2391
  no 1447: 0.2329
  no 1623: 0.2149


## Bloco 3 - Visualizacoes

### 3.1 Graficos

**Implementar:**
- Grafo colorido por comunidade (Bloco 2.1);
- Um grafico por metrica de centralidade (Bloco 2.2), com tamanho/cor do no
  proporcional ao valor e os 10 maiores rotulados (id + valor);
- Salvar as figuras em `../figuras/`.


In [5]:
# codigo desta etapa aqui


## Bloco 4 - Analise

### 4.1 Discussao

Com base nos resultados dos Blocos 2 e 3, responder de forma objetiva:

- Quais nos tem maior influencia, e segundo qual metrica;
- Ha sobreposicao entre os top 10 de metricas diferentes;
- O que a estrutura de comunidades indica sobre a rede;
- O que a fragmentacao da amostra (Bloco 1.2) indica sobre a rede original.

Esta secao alimenta o mini-relatorio (ate 5 paginas, entrega separada no
aprender3.unb.br).
